In [2]:
# create_notebook.py

from pathlib import Path
import nbformat as nbf


NOTEBOOK_DIR = Path("notebooks")
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = NOTEBOOK_DIR / "01_data_preprocessing.ipynb"

nb = nbf.v4.new_notebook()

cells = []


# ============================================================
# TITLE
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
# Task 1 — Data Preprocessing

## UCI Adult Census Income Dataset

**Project:** End-to-End Machine Learning System
**Task:** Data Preprocessing
**Dataset:** UCI Adult / Census Income
**Dataset ID:** UCI Machine Learning Repository #2

---

### Objectives

This notebook implements a production-oriented data preprocessing workflow covering:

1. Authentic dataset acquisition
2. Dataset structure and quality validation
3. Missing-value analysis
4. Exploratory data analysis
5. Outlier detection
6. Outlier treatment
7. Train/test splitting
8. Missing-value imputation
9. Numerical feature scaling
10. Categorical feature encoding
11. Data-leakage prevention
12. Processed dataset generation
13. Professional visualizations
14. Preprocessing pipeline serialization
15. Final preprocessing validation

### Important Design Principle

All statistics required for preprocessing are learned **only from the training dataset**.

This prevents information from the test set leaking into the training process.
"""
    )
)


# ============================================================
# INSTALLATION / IMPORTS
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 1. Environment and Dependencies

The notebook uses:

- pandas
- numpy
- scipy
- scikit-learn
- matplotlib
- seaborn
- joblib
- ucimlrepo

The dataset is retrieved from the official UCI Machine Learning Repository.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
# Uncomment the following line if packages are not installed.

# %pip install -q pandas numpy scipy scikit-learn matplotlib seaborn joblib ucimlrepo
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
from pathlib import Path
import warnings
import json
import joblib

import numpy as np
import pandas as pd
import scipy.sparse as sp

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ucimlrepo import fetch_ucirepo

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
"""
    )
)


# ============================================================
# PROJECT CONFIGURATION
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 2. Project Configuration

A consistent directory structure is used so that the notebook can later become part of an end-to-end production project.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
# ============================================================
# PROJECT DIRECTORIES
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
REPORTS_DIR = OUTPUT_DIR / "reports"

MODELS_DIR = PROJECT_ROOT / "models"

for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    SPLITS_DIR,
    OUTPUT_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
    MODELS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# RANDOM STATE / SPLIT CONFIGURATION
# ============================================================

RANDOM_STATE = 42
TEST_SIZE = 0.20

TARGET_COLUMN = "income"

print(f"Project root: {PROJECT_ROOT}")
print(f"Test size: {TEST_SIZE}")
print(f"Random state: {RANDOM_STATE}")
"""
    )
)


# ============================================================
# PROFESSIONAL VISUAL STYLE
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 3. Professional Visualization Theme

The visualizations use a consistent professional color palette suitable for a project report or presentation.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
COLORS = {
    "navy": "#0B1F33",
    "blue": "#2563EB",
    "cyan": "#06B6D4",
    "green": "#10B981",
    "orange": "#F59E0B",
    "red": "#EF4444",
    "purple": "#7C3AED",
    "slate": "#64748B",
    "light": "#F8FAFC",
    "dark": "#111827",
}

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

plt.rcParams.update({
    "figure.figsize": (12, 7),
    "figure.dpi": 120,
    "axes.titlesize": 17,
    "axes.titleweight": "bold",
    "axes.titlecolor": COLORS["navy"],
    "axes.labelsize": 12,
    "axes.labelcolor": COLORS["dark"],
    "xtick.color": COLORS["slate"],
    "ytick.color": COLORS["slate"],
    "font.family": "DejaVu Sans",
    "legend.frameon": False,
})
"""
    )
)


# ============================================================
# DATA ACQUISITION
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 4. Load the Authentic UCI Adult Dataset

The Adult dataset is obtained through the `ucimlrepo` package using the official UCI dataset identifier.

This avoids depending on an unofficial Kaggle mirror.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
# UCI Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

df = X.copy()
df[TARGET_COLUMN] = y.values

print("=" * 80)
print("UCI ADULT DATASET")
print("=" * 80)
print(f"Rows:       {df.shape[0]:,}")
print(f"Columns:    {df.shape[1]}")
print(f"Features:   {X.shape[1]}")
print(f"Target:     {TARGET_COLUMN}")
"""
    )
)


# ============================================================
# DATASET METADATA
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 5. Dataset Metadata

The UCI metadata is inspected to document the source and dataset structure.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
print("Dataset metadata:")
print(adult.metadata)

print("\nVariable information:")
print(adult.variables)
"""
    )
)


# ============================================================
# INITIAL INSPECTION
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 6. Initial Data Inspection

Before preprocessing, we inspect:

- Shape
- Data types
- First observations
- Numerical summary
- Categorical summary
- Duplicate records
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset information:")
df.info()

print("\nDuplicate rows:", df.duplicated().sum())
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
print("Numerical summary:")
display(df.describe(include=[np.number]).T)
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
print("Categorical summary:")
display(df.describe(include=["object"]).T)
"""
    )
)


# ============================================================
# CLEAN MISSING MARKERS
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 7. Convert Missing-Value Markers

The UCI Adult data may represent missing categorical observations with `?`.

These markers must be converted to proper `NaN` values before missing-value analysis and imputation.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
df_clean = df.copy()

# Convert UCI missing-value marker to NaN
df_clean = df_clean.replace("?", np.nan)

# Remove unnecessary whitespace
object_columns = df_clean.select_dtypes(include=["object"]).columns

for column in object_columns:
    df_clean[column] = df_clean[column].str.strip()

# Clean target labels
df_clean[TARGET_COLUMN] = (
    df_clean[TARGET_COLUMN]
    .astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
)

print("Missing-value markers converted successfully.")
"""
    )
)


# ============================================================
# MISSING VALUE ANALYSIS
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 8. Missing-Value Analysis

Missing values are quantified by:

- Absolute count
- Percentage of observations

This determines the appropriate treatment strategy.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
missing_count = df_clean.isna().sum()
missing_percent = (
    df_clean.isna().mean() * 100
)

missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent.round(2)
})

missing_report = (
    missing_report
    .sort_values("missing_count", ascending=False)
)

display(missing_report)
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
missing_plot = (
    missing_report[
        missing_report["missing_count"] > 0
    ]
)

if not missing_plot.empty:

    plt.figure(figsize=(11, 6))

    ax = sns.barplot(
        data=missing_plot.reset_index(),
        x="missing_count",
        y="index",
        color=COLORS["blue"]
    )

    plt.title(
        "Missing Values by Feature",
        loc="left"
    )

    plt.xlabel("Number of Missing Records")
    plt.ylabel("Feature")

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.0f",
            padding=4
        )

    plt.tight_layout()

    path = FIGURES_DIR / "01_missing_values.png"

    plt.savefig(
        path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    print(f"Saved: {path}")

else:
    print("No missing values detected.")
"""
    )
)


# ============================================================
# TARGET DISTRIBUTION
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 9. Target Distribution

The target variable represents income:

- `<=`50K`
- `>`50K`

The target distribution is inspected before modeling.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
target_counts = df_clean[TARGET_COLUMN].value_counts()

target_percent = (
    df_clean[TARGET_COLUMN]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_report = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percent
})

display(target_report)
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
plt.figure(figsize=(10, 6))

ax = sns.countplot(
    data=df_clean,
    x=TARGET_COLUMN,
    hue=TARGET_COLUMN,
    palette=[
        COLORS["blue"],
        COLORS["green"]
    ],
    legend=False
)

plt.title(
    "Income Target Distribution",
    loc="left"
)

plt.xlabel("Income Class")
plt.ylabel("Number of Records")

for container in ax.containers:
    ax.bar_label(
        container,
        padding=3
    )

plt.tight_layout()

path = FIGURES_DIR / "02_target_distribution.png"

plt.savefig(
    path,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved: {path}")
"""
    )
)


# ============================================================
# FEATURE DEFINITIONS
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 10. Feature-Type Definition

The preprocessing strategy differs according to feature type.

### Numerical features

- age
- fnlwgt
- education-num
- capital-gain
- capital-loss
- hours-per-week

### Categorical features

- workclass
- education
- marital-status
- occupation
- relationship
- race
- sex
- native-country
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
NUMERIC_FEATURES = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]

CATEGORICAL_FEATURES = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

print("Numerical features:")
print(NUMERIC_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)
"""
    )
)


# ============================================================
# NUMERICAL DISTRIBUTIONS
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 11. Numerical Feature Distributions

Histograms with KDE curves are used to inspect:

- Skewness
- Concentration
- Long tails
- Potential extreme observations
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
for column in NUMERIC_FEATURES:

    plt.figure(figsize=(11, 6))

    sns.histplot(
        data=df_clean,
        x=column,
        kde=True,
        color=COLORS["blue"],
        edgecolor="white"
    )

    plt.title(
        f"Distribution of {column}",
        loc="left"
    )

    plt.xlabel(column)
    plt.ylabel("Frequency")

    plt.tight_layout()

    path = (
        FIGURES_DIR /
        f"03_distribution_{column.replace('-', '_')}.png"
    )

    plt.savefig(
        path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
"""
    )
)


# ============================================================
# OUTLIER DETECTION
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 12. Outlier Detection

Outliers are examined using boxplots and the IQR rule.

The IQR rule is:

`Lower Bound = Q1 - 1.5 × IQR`

`Upper Bound = Q3 + 1.5 × IQR`

Important:

> An outlier is not automatically an invalid observation.

For this dataset, potentially legitimate extreme values are retained and handled using percentile-based clipping during preprocessing.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
outlier_summary = []

for column in NUMERIC_FEATURES:

    series = df_clean[column].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (series < lower_bound) |
        (series > upper_bound)
    )

    count = int(outlier_mask.sum())

    percentage = (
        count / len(series) * 100
    )

    outlier_summary.append({
        "feature": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": count,
        "outlier_percentage": percentage,
    })

outlier_report = pd.DataFrame(outlier_summary)

display(
    outlier_report.style.format({
        "Q1": "{:.2f}",
        "Q3": "{:.2f}",
        "IQR": "{:.2f}",
        "lower_bound": "{:.2f}",
        "upper_bound": "{:.2f}",
        "outlier_percentage": "{:.2f}%"
    })
)
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
for column in NUMERIC_FEATURES:

    plt.figure(figsize=(11, 4))

    sns.boxplot(
        x=df_clean[column],
        color=COLORS["cyan"]
    )

    plt.title(
        f"Outlier Analysis — {column}",
        loc="left"
    )

    plt.xlabel(column)

    plt.tight_layout()

    path = (
        FIGURES_DIR /
        f"04_boxplot_{column.replace('-', '_')}.png"
    )

    plt.savefig(
        path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
"""
    )
)


# ============================================================
# CATEGORICAL FEATURES
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 13. Categorical Feature Analysis

The most frequent categories are visualized for each categorical feature.

Only the top 15 categories are shown when a feature has many unique values.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
for column in CATEGORICAL_FEATURES:

    counts = (
        df_clean[column]
        .fillna("Missing")
        .value_counts()
        .head(15)
        .sort_values()
    )

    plt.figure(figsize=(12, 7))

    sns.barplot(
        x=counts.values,
        y=counts.index,
        color=COLORS["purple"]
    )

    plt.title(
        f"Top Categories — {column}",
        loc="left"
    )

    plt.xlabel("Number of Records")
    plt.ylabel(column)

    plt.tight_layout()

    path = (
        FIGURES_DIR /
        f"05_category_{column.replace('-', '_')}.png"
    )

    plt.savefig(
        path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
"""
    )
)


# =================================4============================
# CORRELATION
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 14. Numerical Feature Correlation

A correlation heatmap is used to understand relationships among numerical variables.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
correlation = df_clean[NUMERIC_FEATURES].corr()

plt.figure(figsize=(11, 8))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    square=True
)

plt.title(
    "Numerical Feature Correlation",
    loc="left"
)

plt.tight_layout()

path = FIGURES_DIR / "06_correlation_heatmap.png"

plt.savefig(
    path,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved: {path}")
"""
    )
)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 15. Train/Test Split

The dataset is divided into:

- **80% training**
- **20% testing**

Stratification is used to preserve the target-class distribution.

### Why split before preprocessing?

Imputation values, outlier thresholds, scaling statistics, and categorical encoding must not be learned from the test data.

This is essential for preventing data leakage.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
X = df_clean.drop(columns=[TARGET_COLUMN])

y = (
    df_clean[TARGET_COLUMN]
    .map({
        "<=50K": 0,
        ">50K": 1
    })
)

if y.isna().any():
    raise ValueError(
        "Unexpected target labels detected."
    )

y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining target distribution:")
display(
    y_train.value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)

print("\nTesting target distribution:")
display(
    y_test.value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)
"""
    )
)


# ============================================================
# CUSTOM OUTLIER CLIPPER
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 16. Training-Only Quantile Outlier Clipper

Instead of deleting rows, numerical values are clipped to the 1st and 99th percentiles.

The bounds are learned **only from the training data**.

This makes the transformation leakage-safe and robust for future inference.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        '''
class QuantileClipper(BaseEstimator, TransformerMixin):
    """
    Clip numerical values using quantiles learned during fit.
    """

    def __init__(
        self,
        lower_quantile=0.01,
        upper_quantile=0.99
    ):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):

        X_df = pd.DataFrame(X).copy()

        self.feature_names_in_ = (
            X_df.columns.tolist()
        )

        self.lower_bounds_ = (
            X_df.quantile(self.lower_quantile)
        )

        self.upper_bounds_ = (
            X_df.quantile(self.upper_quantile)
        )

        return self

    def transform(self, X):

        X_df = pd.DataFrame(
            X,
            columns=self.feature_names_in_
        )

        clipped = X_df.clip(
            lower=self.lower_bounds_,
            upper=self.upper_bounds_,
            axis="columns"
        )

        return clipped.to_numpy()
'''
    )
)


# ============================================================
# PREPROCESSING PIPELINE
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 17. Build the Leakage-Safe Preprocessing Pipeline

### Numerical pipeline

1. Median imputation
2. 1st–99th percentile outlier clipping
3. Standard scaling

### Categorical pipeline

1. Most-frequent imputation
2. One-hot encoding

The entire transformation is wrapped in a `ColumnTransformer`.
"""
    )
)

cells.append(
    nbf.v4.new_code_cell(
        """
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "outlier_clipper",
            QuantileClipper(
                lower_quantile=0.01,
                upper_quantile=0.99
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        ),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        ),
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")
"""
    )
)


# ============================================================
# FIT ONLY ON TRAIN
# ============================================================

cells.append(
    nbf.v4.new_markdown_cell(
        """
## 18. Fit and Transform

**Critical rule:**

```text
fit_transform(train)
transform(test)
```
The test data is never used when calculating preprocessing parameters.
"""
)
)

cells.append(
nbf.v4.new_code_cell(
"""
Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(
X_train,
y_train
)

# Transform test data using parameters learned from training
X_test_processed = preprocessor.transform(
X_test
)

print(
"Processed training shape:",
X_train_processed.shape
)

print(
"Processed testing shape:",
X_test_processed.shape
)
"""
)
)

# ============================================================
# FEATURE NAMES
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## 19. Extract Processed Feature Names
"""
)
)

cells.append(
nbf.v4.new_code_cell(
"""
feature_names = (
preprocessor
.get_feature_names_out()
)

print(
f"Number of processed features: "
f"{len(feature_names):,}"
)

print("\nFirst 30 processed features:")

for name in feature_names[:30]:
    print(name)
"""
)
)

# ============================================================
# PROCESSED DATA VALIDATION
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## 20. Validate the Processed Data
We verify that:

- Train and test have identical feature dimensions
- Numerical scaling has been applied
- Categorical variables have been encoded
- No missing values remain
- Unknown categories are handled safely
"""
)
)
cells.append(
nbf.v4.new_code_cell(
"""
assert (
X_train_processed.shape[1]
== X_test_processed.shape[1]
), "Train/test feature mismatch."

assert not np.isnan(
X_train_processed.toarray()
if sp.issparse(X_train_processed)
else X_train_processed
).any(), "NaN values found in training matrix."

assert not np.isnan(
X_test_processed.toarray()
if sp.issparse(X_test_processed)
else X_test_processed
).any(), "NaN values found in testing matrix."

print("✓ Train/test feature dimensions match.")
print("✓ No NaN values in processed training data.")
print("✓ No NaN values in processed testing data.")
print("✓ Validation successful.")
"""
)
)

# ============================================================
# NUMERICAL SCALING VALIDATION
# ============================================================
cells.append(
nbf.v4.new_code_cell(
"""
# Validate the transformed numerical features.
numeric_transformer = (
preprocessor
.named_transformers_["numeric"]
)

numeric_feature_names = NUMERIC_FEATURES

numeric_processed = (
numeric_transformer.transform(
X_train[NUMERIC_FEATURES]
)
)

numeric_processed_df = pd.DataFrame(
numeric_processed,
columns=numeric_feature_names
)

scaling_validation = pd.DataFrame({
"mean": numeric_processed_df.mean(),
"std": numeric_processed_df.std()
})

display(
scaling_validation.style.format({
"mean": "{:.4f}",
"std": "{:.4f}"
})
)
"""
)
)

# ============================================================
# SAVE RAW SPLITS
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## 21. Save Train/Test Splits
Raw train/test feature partitions are saved so the exact experimental split can be reproduced.
"""
)
)

cells.append(
nbf.v4.new_code_cell(
"""
train_raw = X_train.copy()
train_raw[TARGET_COLUMN] = y_train.values

test_raw = X_test.copy()
test_raw[TARGET_COLUMN] = y_test.values

train_raw_path = SPLITS_DIR / "train_raw.csv"
test_raw_path = SPLITS_DIR / "test_raw.csv"

train_raw.to_csv(
train_raw_path,
index=False
)

test_raw.to_csv(
test_raw_path,
index=False
)

print(f"Saved: {train_raw_path}")
print(f"Saved: {test_raw_path}")
"""
)
)

# ============================================================
# SAVE PROCESSED MATRICES
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## 22. Save Processed Matrices
Because one-hot encoding creates a potentially large sparse matrix, processed feature matrices are saved in .npz sparse format.
"""
)
)

cells.append(
nbf.v4.new_code_cell(
"""
X_train_path = SPLITS_DIR / "X_train_processed.npz"
X_test_path = SPLITS_DIR / "X_test_processed.npz"

y_train_path = SPLITS_DIR / "y_train.csv"
y_test_path = SPLITS_DIR / "y_test.csv"

sp.save_npz(
X_train_path,
X_train_processed
)

sp.save_npz(
X_test_path,
X_test_processed
)

pd.DataFrame({
TARGET_COLUMN: y_train
}).to_csv(
y_train_path,
index=False
)

pd.DataFrame({
TARGET_COLUMN: y_test
}).to_csv(
y_test_path,
index=False
)

print(f"Saved: {X_train_path}")
print(f"Saved: {X_test_path}")
print(f"Saved: {y_train_path}")
print(f"Saved: {y_test_path}")
"""
)
)

# ============================================================
# SAVE FEATURE NAMES
# ============================================================
cells.append(
nbf.v4.new_code_cell(
"""
feature_names_path = (
PROCESSED_DIR /
"processed_feature_names.csv"
)

pd.DataFrame({
"feature_name": feature_names
}).to_csv(
feature_names_path,
index=False
)

print(f"Saved: {feature_names_path}")
"""
)
)

# ============================================================
# SAVE PREPROCESSOR
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## 23. Serialize the Preprocessing Pipeline
The fitted pipeline is saved with joblib.

This is critical for production because real-time inference must use exactly the same preprocessing transformations as model training.
"""
)
)

cells.append(
nbf.v4.new_code_cell(
"""
pipeline_path = (
MODELS_DIR /
"preprocessing_pipeline.joblib"
)

joblib.dump(
preprocessor,
pipeline_path
)

print(
f"Preprocessing pipeline saved to:\n"
f"{pipeline_path}"
)
"""
)
)

# ============================================================
# SAVE SUMMARY
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## 24. Generate Preprocessing Summary
A machine-readable summary is generated for reporting and reproducibility.
"""
)
)

cells.append(
nbf.v4.new_code_cell(
"""
summary = pd.DataFrame({
"metric": [
"original_rows",
"original_columns",
"feature_columns",
"training_rows",
"testing_rows",
"test_size",
"original_missing_cells",
"cleaned_missing_cells",
"processed_feature_count",
"train_positive_class_ratio",
"test_positive_class_ratio",
"random_state",
],
"value": [
len(df),
df.shape[1],
X.shape[1],
len(X_train),
len(X_test),
TEST_SIZE,
int(df.isna().sum().sum()),
int(df_clean.isna().sum().sum()),
len(feature_names),
round(y_train.mean(), 6),
round(y_test.mean(), 6),
RANDOM_STATE,
]
})

summary_path = (
REPORTS_DIR /
"preprocessing_summary.csv"
)

summary.to_csv(
summary_path,
index=False
)

display(summary)

print(f"Saved: {summary_path}")
"""
)
)

# ============================================================
# FINAL ARTIFACT CHECK
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## 25. Final Artifact Verification
"""
)
)

cells.append(
nbf.v4.new_code_cell(
"""
expected_files = [
FIGURES_DIR / "01_missing_values.png",
FIGURES_DIR / "02_target_distribution.png",
FIGURES_DIR / "06_correlation_heatmap.png",

SPLITS_DIR / "train_raw.csv",
SPLITS_DIR / "test_raw.csv",
SPLITS_DIR / "X_train_processed.npz",
SPLITS_DIR / "X_test_processed.npz",
SPLITS_DIR / "y_train.csv",
SPLITS_DIR / "y_test.csv",

PROCESSED_DIR / "processed_feature_names.csv",

REPORTS_DIR / "preprocessing_summary.csv",

MODELS_DIR / "preprocessing_pipeline.joblib",

]

verification = []

for path in expected_files:

    verification.append({
        "file": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size
        if path.exists()
        else 0
    })

verification_df = pd.DataFrame(verification)

display(verification_df)

if verification_df["exists"].all():
    print("\n✓ All expected preprocessing artifacts exist.")
else:
    missing = verification_df.loc[
        ~verification_df["exists"],
        "file"
    ].tolist()

    raise FileNotFoundError(
        f"Missing artifacts: {missing}"
    )

"""
)
)

# ============================================================
# FINAL SUMMARY
# ============================================================
cells.append(
nbf.v4.new_markdown_cell(
"""
## Task 1 — Final Result
Preprocessing completed successfully.

The project now has a reproducible and deployment-oriented preprocessing workflow.

### Completed requirements
Requirement	Status
Authentic dataset	✓
Data inspection	✓
Missing-value analysis	✓
Missing-value handling	✓
Outlier detection	✓
Outlier treatment	✓
Numerical scaling	✓
Categorical encoding	✓
Train/test split	✓
Stratification	✓
Data leakage prevention	✓
Professional visualizations	✓
Processed datasets	✓
Feature names	✓
Serialized preprocessing pipeline	✓
Reproducible random state	✓
Validation checks	✓

### Production Flow
UCI Dataset
    ↓
Data Validation
    ↓
Missing-Value Detection
    ↓
Train/Test Split
    ↓
Training-Only Preprocessing
    ├── Median Imputation
    ├── Quantile Outlier Clipping
    ├── Standard Scaling
    ├── Categorical Imputation
    └── One-Hot Encoding
    ↓
Processed Features
    ↓
Saved Pipeline
    ↓
Ready for Model Training

The serialized preprocessing pipeline can now be reused by the subsequent model-training and real-time inference components.
"""
)
)

nb["cells"] = cells

nb["metadata"] = {
"kernelspec": {
"display_name": "Python 3",
"language": "python",
"name": "python3",
},
"language_info": {
"name": "python",
"version": "3.x",
},
}

nbf.write(nb, NOTEBOOK_PATH)

print("=" * 80)
print("NOTEBOOK CREATED SUCCESSFULLY")
print("=" * 80)
print(f"Path: {NOTEBOOK_PATH.resolve()}")
print(f"Cells: {len(nb.cells)}")

# ============================================================
# CREATE THE ACTUAL NOTEBOOK
# ============================================================

NOTEBOOK CREATED SUCCESSFULLY
Path: /content/notebooks/01_data_preprocessing.ipynb
Cells: 60
